# Residency Day 3: Project Deliverable 4

## Final insights, recommendations, and presentation-ready analysis

This notebook consolidates the end-to-end workflow for the CMS healthcare claims project. It brings together the data preparation, regression, classification, clustering, and association-rule mining results into a single, presentation-ready summary that connects the technical work to the healthcare business question.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

# Define the project paths so the notebook can be run from the workspace root.
DATA_PATH = Path('cms_healthcare_claims_cleaned.csv')
output_dir = Path('deliverable_3_outputs')
output_dir.mkdir(exist_ok=True)

# Load the cleaned dataset and the saved outputs from Deliverable 3.
df = pd.read_csv(DATA_PATH)
classification_results = pd.read_csv(output_dir / 'classification_results.csv')
cluster_summary = pd.read_csv(output_dir / 'cluster_summary.csv')
association_rules = pd.read_csv(output_dir / 'association_rules.csv')

# Display a quick preview to confirm the data loaded correctly.
df.head()


In [ ]:
# ------------------------------------------------------------------
# Section 1: Prepare the final analysis features
# ------------------------------------------------------------------
# These engineered features are used throughout the final summary so the
# notebook can connect raw claims data with the modeling outputs.
df = df.copy()

df['Year_Value'] = pd.to_numeric(df['Year'].str.extract(r'(\d{4})')[0], errors='coerce')
df['Quarter'] = df['Year'].str.extract(r'(Q[1-4])').fillna('Q1').iloc[:, 0].str.replace('Q', '').astype(int)
df['Claims_per_Beneficiary'] = df['Tot_Clms'] / df['Tot_Benes'].replace(0, np.nan)
df['Log_Tot_Spending'] = np.log1p(df['Tot_Spndng'])

# Create a simple high-utilization flag to support the final narrative.
df['High_Claims'] = (df['Tot_Clms'] > df['Tot_Clms'].quantile(0.75)).astype(int)
df['High_Spending'] = (df['Tot_Spndng'] > df['Tot_Spndng'].quantile(0.75)).astype(int)
df['High_Beneficiaries'] = (df['Tot_Benes'] > df['Tot_Benes'].quantile(0.75)).astype(int)

print('Dataset overview:')
print(df[['Tot_Benes', 'Tot_Clms', 'Tot_Spndng', 'Avg_Spnd_Per_Bene', 'Avg_Spnd_Per_Clm', 'Claims_per_Beneficiary', 'Log_Tot_Spending']].describe().round(2))


In [ ]:
# ------------------------------------------------------------------
# Section 2: Visualize the main findings for the final presentation
# ------------------------------------------------------------------
# The plots below turn the modeling outputs into a clear story for the report.
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: spending grows with claims volume, a key business insight.
axes[0].scatter(df['Tot_Clms'], df['Tot_Spndng'], alpha=0.6, color='royalblue')
axes[0].set_title('Claims Volume vs Spending')
axes[0].set_xlabel('Total Claims')
axes[0].set_ylabel('Total Spending')

# Panel 2: classification performance shows which methods were most reliable.
axes[1].bar(classification_results['Model'], classification_results['F1'], color='steelblue')
axes[1].set_title('Classification F1 Scores')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('F1 Score')
axes[1].tick_params(axis='x', rotation=30)

# Panel 3: clustering shows the distribution of utilization groups.
axes[2].bar(cluster_summary['Cluster'].astype(str), cluster_summary['Count'], color='seagreen')
axes[2].set_title('Cluster Sizes')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()


In [ ]:
# ------------------------------------------------------------------
# Section 3: Synthesize the most important project conclusions
# ------------------------------------------------------------------
# These points are written in a presentation-friendly form and connect the
# technical results back to the healthcare motivation of the project.

best_model = classification_results.loc[classification_results['F1'].idxmax(), 'Model']
best_f1 = classification_results.loc[classification_results['F1'].idxmax(), 'F1']

# Pull out the strongest association-rule insight from the Deliverable 3 outputs.
strongest_rule = association_rules.sort_values('lift', ascending=False).iloc[0]

summary_points = [
    f'The strongest classification model was {best_model} with an F1 score of {best_f1:.3f}.',
    'The relationship between claims volume and total spending is clearly positive, which supports the value of utilization-based monitoring.',
    'The clustering output separated groups with distinct spending and beneficiary patterns, helping explain heterogeneity in the dataset.',
    f'The strongest association rule showed a lift of {strongest_rule["lift"]:.3f}, indicating that high-utilization indicators often occur together.',
    'Overall, the project demonstrates that combining predictive modeling with descriptive analytics provides a stronger understanding of healthcare spending behavior.'
]

for item in summary_points:
    print('-', item)

# A compact table for the final report summary.
summary_table = pd.DataFrame({
    'Metric': ['Best classification model', 'Best F1 score', 'Largest cluster count', 'Strongest rule lift'],
    'Value': [best_model, f'{best_f1:.3f}', int(cluster_summary['Count'].max()), f'{strongest_rule["lift"]:.3f}']
})

summary_table
